# Sémio-1 : entraîner ton propre modèle médical

Ce notebook transforme le modèle open source Gemma 4 E2B en **Sémio-1**, ton modèle médical, puis le publie sur **ton** compte Hugging Face pour que ton site le charge.

**Avant de commencer**
1. Menu **Exécution → Modifier le type d'exécution → GPU T4**, puis **Enregistrer**.
2. Crée un compte sur huggingface.co, puis un **token d'accès en écriture** : *Settings → Access Tokens → Create new token → Write*.
3. Lance les cellules une par une, de haut en bas (▶). Compte environ **1 h 30** au total.

Données utilisées (licences ouvertes) :
- **MediQAl** (CC BY 4.0) : environ 16 000 QCM d'ECN en français, avec les réponses.
- **FrenchMedMCQA** (Apache 2.0) : environ 2 000 QCM de pharmacie.
- **Tes propres données** (optionnel) : tes QCM ou cas corrigés, dans un fichier CSV ou JSONL.


## 1. Installation (2 à 4 min)

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --upgrade huggingface_hub
import torch; print(torch.cuda.get_device_name(0))

## 2. Réglages
Change `HF_USER` par ton nom d'utilisateur Hugging Face.

In [ ]:
HF_USER   = "ton-nom-hf"          # <-- ton nom d'utilisateur Hugging Face
REPO_NAME = "semio-1-E2B-GGUF"    # nom du dépôt du modèle
AUTEUR    = "Yahya"
MAX_STEPS = 400                   # ~1 h sur T4. Plus = meilleur mais plus long.
N_EVAL    = 150                   # nombre de QCM pour mesurer le score avant/après
MAX_SEQ   = 1024

## 3. Connexion à Hugging Face
Colle **toi-même** ton token *Write* dans la case qui apparaît.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Charger le modèle de base

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    max_seq_length = MAX_SEQ,
    load_in_4bit = True,
    full_finetuning = False,
)
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-4")

## 5. Préparer les données

In [ ]:
import json, gzip, random, re
import pandas as pd
from huggingface_hub import hf_hub_download
random.seed(3407)

SYSTEM = (f"Tu es Sémio, l'assistant médical créé par {AUTEUR} pour les étudiants en médecine. "
          "Tu réponds en français, de façon exacte et pédagogique.")

def read_json_any(path):
    op = gzip.open if path.endswith(".gz") else open
    with op(path, "rt", encoding="utf-8") as f:
        txt = f.read().strip()
    if txt.startswith("["):
        return json.loads(txt)
    if txt.startswith("{") and "\n{" not in txt[:5000]:
        obj = json.loads(txt)
        for k in ("data", "rows", "questions"):
            if isinstance(obj, dict) and k in obj: return obj[k]
        return [obj]
    return [json.loads(l) for l in txt.splitlines() if l.strip()]

def qcm_prompt(question, options, case=None, multi=False):
    s = ""
    if case: s += f"Cas clinique : {case}\n\n"
    s += f"Question : {question}\n"
    for k in "ABCDE":
        if options.get(k): s += f"{k}. {options[k]}\n"
    s += "\n" + ("Plusieurs réponses peuvent être justes. " if multi else "Une seule réponse est juste. ")
    s += "Donne la ou les bonnes réponses."
    return s

def qcm_answer(letters, options):
    letters = [l for l in letters if l in options]
    lines = [f"- **{l}.** {options[l]}" for l in letters]
    head = "Bonne réponse" if len(letters) == 1 else "Bonnes réponses"
    return f"{head} : **{', '.join(letters)}**\n" + "\n".join(lines)

def conv(user, assistant):
    return {"conversations": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant}]}

rows, eval_rows = [], []

# --- MediQAl (ECN, CC BY 4.0) ---
for cfg in ("mcqu", "mcqm"):
    for split in ("train", "validation"):
        p = hf_hub_download("ANR-MALADES/MediQAl", f"{cfg}/{split}.json", repo_type="dataset")
        for r in read_json_any(p):
            opts = {k.upper(): (r.get(f"answer_{k}") or "").strip() for k in "abcde"}
            good = [x.strip().upper() for x in str(r.get("correct_answers", "")).split(",") if x.strip()]
            if not good: continue
            item = (qcm_prompt(r["question"], opts, r.get("clinical_case"), multi=(cfg == "mcqm")), good, opts)
            if split == "validation" and cfg == "mcqu": eval_rows.append(item)
            elif split == "train": rows.append(conv(item[0], qcm_answer(good, opts)))
print("MediQAl :", len(rows), "exemples d'entraînement,", len(eval_rows), "QCM d'évaluation")

# --- FrenchMedMCQA-extended (pharmacie) ---
try:
    p = hf_hub_download("uy-rrodriguez/FrenchMedMCQA-extended", "train.json.gz", repo_type="dataset")
    n0 = len(rows)
    for r in read_json_any(p):
        ans = r.get("answers") or {}
        opts = {k.upper(): (ans.get(k) or "").strip() for k in "abcde"}
        good = [x.strip().upper() for x in (r.get("correct_answers") or [])]
        if good:
            rows.append(conv(qcm_prompt(r["question"], opts, multi=len(good) > 1), qcm_answer(good, opts)))
    print("FrenchMedMCQA :", len(rows) - n0, "exemples")
except Exception as e:
    print("FrenchMedMCQA ignoré :", e)

# --- Identité de Sémio ---
IDENTITE = [
    ("Qui es-tu ?", f"Je suis Sémio, un assistant médical créé par {AUTEUR} pour aider les étudiants en médecine."),
    ("Qui t'a créé ?", f"J'ai été créé par {AUTEUR}. Je m'appelle Sémio."),
    ("Comment tu t'appelles ?", "Je m'appelle Sémio."),
    ("Tu es quel modèle ?", f"Je suis Sémio-1, le modèle de Sémio, créé par {AUTEUR}."),
]
for q, a in IDENTITE:
    for _ in range(8): rows.append(conv(q, a))

print("Total :", len(rows))

## 6. (Optionnel) Ajouter tes propres données
Clique sur l'icône **dossier** à gauche et dépose un fichier `mes_donnees.csv` avec deux colonnes : `question` et `reponse`. Ce sont tes QCM corrigés ou tes cas cliniques corrigés. Sinon, passe cette cellule.

In [ ]:
import os
if os.path.exists("mes_donnees.csv"):
    df = pd.read_csv("mes_donnees.csv")
    for _, r in df.iterrows():
        if isinstance(r["question"], str) and isinstance(r["reponse"], str):
            for _ in range(2): rows.append(conv(r["question"], r["reponse"]))   # 2x : tes données comptent plus
    print("Tes données ajoutées :", len(df))
else:
    print("Pas de fichier mes_donnees.csv : on continue sans.")

In [ ]:
from datasets import Dataset
random.shuffle(rows)
dataset = Dataset.from_list(rows)
def fmt(ex):
    return {"text": tokenizer.apply_chat_template(ex["conversations"], tokenize=False,
                                                  add_generation_prompt=False).removeprefix("<bos>")}
dataset = dataset.map(fmt, remove_columns=["conversations"])
print(dataset[0]["text"][:800])

## 7. Score AVANT entraînement
On mesure le pourcentage de bonnes réponses sur des QCM d'ECN que le modèle n'a jamais vus.

In [ ]:
random.shuffle(eval_rows)
EVAL = eval_rows[:N_EVAL]

def score(model, n=N_EVAL):
    ok = 0
    for prompt, good, _ in EVAL[:n]:
        msgs = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": prompt + "\nRéponds uniquement par la lettre."}]
        text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        inp = tokenizer(text=[text], return_tensors="pt").to("cuda")
        out = model.generate(**inp, max_new_tokens=12, do_sample=False)
        rep = tokenizer.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0]
        m = re.findall(r"\b([A-E])\b", rep.upper())
        ok += bool(m) and m[0] == good[0]
    return 100 * ok / n

avant = score(model)
print(f"Score AVANT : {avant:.1f} %")

## 8. Entraînement (environ 1 h)
La perte (*loss*) peut commencer haut (10 à 15) : c'est normal pour ce modèle. Elle doit descendre.

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers = False, finetune_language_layers = True,
    finetune_attention_modules = True, finetune_mlp_modules = True,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none", random_state = 3407,
)
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text", max_seq_length = MAX_SEQ,
        per_device_train_batch_size = 1, gradient_accumulation_steps = 4,
        warmup_steps = 10, max_steps = MAX_STEPS, learning_rate = 2e-4,
        logging_steps = 10, optim = "adamw_8bit", weight_decay = 0.01,
        lr_scheduler_type = "linear", seed = 3407, report_to = "none", output_dir = "outputs",
    ),
)
trainer = train_on_responses_only(trainer, instruction_part = "<|turn>user\n", response_part = "<|turn>model\n")
trainer.train()

## 9. Score APRÈS entraînement

In [ ]:
apres = score(model)
print(f"Score AVANT : {avant:.1f} %   →   Score APRÈS : {apres:.1f} %")

## 10. Essai rapide

In [ ]:
msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "Qui es-tu ?"}]
text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
inp = tokenizer(text=[text], return_tensors="pt").to("cuda")
print(tokenizer.batch_decode(model.generate(**inp, max_new_tokens=60)[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0])

## 11. Export pour le navigateur (GGUF, 10 à 20 min)
Le modèle est compressé (Q4_K_M) puis découpé en morceaux de 512 Mo, comme l'exige le moteur du site.

In [ ]:
model.save_pretrained_gguf("semio_gguf", tokenizer, quantization_method = "q4_k_m")
import glob, os
ggufs = [f for f in glob.glob("semio_gguf*/**/*.gguf", recursive=True) + glob.glob("*.gguf") if "mmproj" not in f.lower()]
ggufs.sort(key=os.path.getsize, reverse=True)
SRC = ggufs[0]; print("Fichier :", SRC, round(os.path.getsize(SRC)/1e9, 2), "Go")

In [ ]:
import shutil, subprocess
split = shutil.which("llama-gguf-split") or next(iter(glob.glob("/root/.unsloth/**/llama-gguf-split", recursive=True) + glob.glob("llama.cpp/**/llama-gguf-split", recursive=True)), None)
if not split:
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
    !cmake -S /content/llama.cpp -B /content/llama.cpp/build -DLLAMA_CURL=OFF > /dev/null
    !cmake --build /content/llama.cpp/build --target llama-gguf-split -j 4 > /dev/null
    split = "/content/llama.cpp/build/bin/llama-gguf-split"
os.makedirs("upload", exist_ok=True)
subprocess.run([split, "--split-max-size", "512M", SRC, "upload/semio-1-E2B-Q4_K_M"], check=True)
print(sorted(os.listdir("upload")))

## 12. Publier Sémio-1 sur ton Hugging Face

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
repo = f"{HF_USER}/{REPO_NAME}"
api.create_repo(repo, exist_ok=True)
open("upload/README.md", "w").write(f"""---
license: apache-2.0
language: [fr]
base_model: google/gemma-4-E2B-it
tags: [medical, french, gguf]
---
# Sémio-1 (E2B)

Modèle médical pour étudiants, créé par {AUTEUR}. Affiné sur MediQAl (CC BY 4.0) et FrenchMedMCQA (Apache 2.0).
Basé sur Gemma 4 E2B (Apache 2.0). Outil pédagogique : ne remplace pas un avis médical.

Score sur {N_EVAL} QCM d'ECN jamais vus : {avant:.1f} % avant → {apres:.1f} % après.
""")
api.upload_folder(folder_path="upload", repo_id=repo)
first = sorted(f for f in os.listdir("upload") if f.endswith(".gguf"))[0]
print("Publié : https://huggingface.co/" + repo)
print("Fichier à donner à Claude pour le site :", first)

Terminé ! Envoie à Claude le **lien du dépôt** et le **nom du premier fichier** affichés au-dessus : il branchera Sémio-1 sur ton site.